<a href="https://colab.research.google.com/github/Wosstarot/CSCI164/blob/main/CSCI164_Search_KevinTigson.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Kevin Tigson

CSCI164

2 May 2025

In [232]:
import random
import heapq
import pandas as pd


# Tile Sliding Domain: Initial State Space

In [233]:
# Global Variables
StateDimension=4                                                                #Change
InitialState=   [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0]                       #Change
GoalState=      [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0]                         #Change
Actions = lambda s: ['u', 'd', 'l', 'r']
Opposite=dict([('u','d'),('d','u'),('l','r'),('r','l'), (None, None)])

# Random Walk Steps
randomWalkSteps = [5,10,20,40,80]

Extend the 3x3 puzzle size to 4x4

In [234]:
#Change the initial state space size (3x3 or 4x4)
def setInitialStateSpaceSize(bool_fourbyfour):                               #Boolean parameter: If true, then 4x4. If false, then 3x3.
  global StateDimension, InitialState, GoalState

# 4x4
  if bool_fourbyfour:
    StateDimension = 4
    InitialState = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0]
    GoalState=[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0]
# 3x3
  else:
    StateDimension = 3
    InitialState = [1,2,3,4,5,6,7,8,0]
    GoalState=[1,2,3,4,5,6,7,8,0]


The **Result Function** creates a new state from the actions and the state we're currently in.

In [235]:
def Result(state, action):
  i = state.index(0)
  newState = list(state)
  row,col=i//StateDimension, i % StateDimension
  if ( (action=='u' and row==0) or
       (action=='d' and row==StateDimension-1) or
       (action=='l' and col==0) or
       (action=='r' and col==StateDimension-1)):
      return newState
  if action=='u':
    l,r = row*StateDimension+col, (row-1)*StateDimension+col
  elif action=='d':
    l,r = row*StateDimension+col, (row+1)*StateDimension+col
  elif action=='l':
    l,r = row*StateDimension+col, row*StateDimension+col-1
  elif action=='r' :
    l,r = row*StateDimension+col, row*StateDimension+col+1
  newState[l], newState[r] = newState[r], newState[l]
  return newState

def PrintState(s):
  for i in range(0,len(s),StateDimension):
    print(s[i:i+StateDimension])

def LegalMove(state, action):
  i = state.index(0)
  row,col=i//StateDimension, i % StateDimension
  newState = state.copy()
  if ( (action=='u' and row==0) or
       (action=='d' and row==StateDimension-1) or
       (action=='l' and col==0) or
       (action=='r' and col==StateDimension-1)):
      return False
  return True


In [236]:
def SingleTileManhattanDistance(tile, left, right):
  leftIndex = left.index(tile)
  rightIndex = right.index(tile)
  return (abs(leftIndex//StateDimension-rightIndex//StateDimension) +
          abs(leftIndex%StateDimension-rightIndex%StateDimension))

def ManhattanDistance(left, right):
  distances = [SingleTileManhattanDistance(tile, left, right)
     for tile in range(1, StateDimension**2)]
  ### print ("Distances= ", distances)
  return sum(distances)


In [237]:
def OutOfPlace(left, right):
  distances = [left[i]!=right[i] and right[i] != 0
     for i in range(StateDimension**2)]
  return sum(distances)

# Random Walk

Take some random moves from a state and return the new state and the sequence of moves.

Optimizations to prevent preserve difficulty: Do not include moves undoing last move, or having no effect.

In [238]:
def RandomWalk(state, steps):
  actionSequence = []
  actionLast = None
  for i in range(steps):
    action = None
    while action==None:
      action = random.choice(Actions(state))
      action = action if (LegalMove(state, action)
          and action!= Opposite[actionLast]) else None
    actionLast = action
    state = Result(state, action)
    actionSequence.append(action)
  return state, actionSequence



Test Random Walk

The function ApplyMoves applies a sequence of actions to go from the initial state to the final state. Can combine with reverse moves to go from the final state to the initial state.

In [239]:
def ApplyMoves(actions, state):
  for action in actions:
    state = Result(state, action)
  return state

ReverseMoves function takes a sequence of actions and reverses them. If a solution is provided, then the solution should return to the initial state. Essentially takes the opposite direction of all elements then reverses the order of all elements in the list.

In [240]:
def ReverseMoves(actions):
  ret = [Opposite[a] for a in actions]
  ret.reverse()
  return ret

## Problem Class

INITIAL = InitialState  
IsGoal = Goal Test  
Actions = Actions List  
Result = Action Behavior  
ActionCost = Action Cost  

Functions passed dynamically

In [241]:
class Problem(object): pass

## Node

Creates a new node object:
  __init__ initializes the Node like a constructor

Similar to a vertex in a local graph. New nodes are linked to the previous node. Tree of states that is connected by actions. Used to construct solution

In [242]:
class Node(object):
  def __init__(self, state, parent=None, action=None, path_cost=0 ):
    self.State=state
    self.Parent=parent
    self.Action=action
    self.PathCost = path_cost

  def __str__(self):
    action = "<none>" if not self.Action else self.Action
    return str(self.State) + ", " + action
  def __repr__(self):
    action = "<none>" if not self.Action else self.Action
    return str(self.State) + ", " + action
  def __lt__(self, other):
    return self.PathCost < other.PathCost;

## Expand

When an action occurs, the expand function obtains a new state.
  Increments the path cost by 1.

Creates a new state from the action and the state we’re currently in.

In [243]:
def Expand(problem, node):
  ret = []
  s = node.State
  for action in problem.Actions(s):
    sPrime = problem.Result(s, action)
    cost =node.PathCost + problem.ActionCost(s,action,sPrime)
    ret.append(Node(sPrime, node, action, cost))
  return ret


## Breadth-First Search

Breadth-First Search function expands in a FIFO queue.
Function used by Breadth-First Search and Depth-First Search
Breadth-First Search considers states that we can get to with the fewest number of actions first. If we take an action and get to a state then we try those states in the same order we were able to generate them. When the nodesExpanded variable is greater than 500,000 the program stops solving the problem.  

In [244]:
def BreadthFirstSearch(problem):
  node = Node(tuple(problem.INITIAL))
  if problem.IsGoal(node.State):
    return node, 0
  Frontier = []
  Frontier.append(node)
  reached = set()
  reached.add(tuple(problem.INITIAL))
  nodesExpanded = 0
  while (Frontier):
    ### print([str(n) for n in Frontier])
    node = Frontier.pop(0)
    ### print(node)
    for child in Expand(problem, node):
      s = tuple(child.State)
      ### print (s, "IsGoal=", problem.IsGoal(s))
      if problem.IsGoal(s):
        return child, nodesExpanded
      if s not in reached:
        reached.add(s)
        Frontier.append(child)
    nodesExpanded += 1
    if nodesExpanded > 500000:
      break;
  return None, nodesExpanded

## Best-First Search

Best-First Search function expands in a priority queue. Function used by Greedy, Uniform Cost Search, and A* w/Heuristics. When the nodesExpanded variable is greater than 500,000 the program stops solving the problem.  

In [245]:
def BestFirstSearch(problem, f):
  node = Node(tuple(problem.INITIAL))
  Frontier = []
  heapq.heappush(Frontier,(f(node), node))
  reached = {}
  reached[tuple(problem.INITIAL)]=node
  nodesExpanded = 0
  while (Frontier):
    ##print([(x, str(n)) for (x,n) in Frontier])
    fValue, node = heapq.heappop(Frontier)
    ##print (node.State, "IsGoal=", problem.IsGoal(tuple(node.State)))
    if problem.IsGoal(tuple(node.State)):
      return node, nodesExpanded    ### print(node)
    for child in Expand(problem, node):
      s = tuple(child.State)
      if s not in reached or child.PathCost < reached[s].PathCost:
        reached[s] = child
        heapq.heappush(Frontier, (f(child), child))
    nodesExpanded += 1
    if nodesExpanded > 500000:
      break;
  return None, nodesExpanded

Set up the 3x3 Sliding Puzzle Domain
  * Assign member variable called initial as the initial state
  * Goal function for when they’re all in line
    * use parenthesis as tuple, when its parenthesis its an immutable list.
  * Assign actions functions can assign it as a member variable as well
  * Same thing with the result function
  * As well as the action cost
    * All actions have a cost of one. so when we pass in a state, action, new state, it always has a cost of one. So we use a lambda function and assign that lambda function straight to the new class object.


In [246]:
setInitialStateSpaceSize(False)
TileSliding = Problem()
TileSliding.INITIAL = InitialState
TileSliding.IsGoal = lambda s: s==(1,2,3,4,5,6,7,8,0)           #Change
TileSliding.Actions = Actions
TileSliding.Result=Result
TileSliding.ActionCost = lambda s, a, sPrime: 1
print( TileSliding.IsGoal((1,2,3,4,5,6,7,8,0)) )                #Change
print( Node(InitialState) )
print(1+TileSliding.ActionCost(1,2,3))                          #Checks if the action cost is always 1 even if 1, two, or 3 is passed.

True
[1, 2, 3, 4, 5, 6, 7, 8, 0], <none>
2


Set up the 4x4 Sliding Puzzle Domain

In [247]:
setInitialStateSpaceSize(True)
TileSliding_4x4 = Problem()
TileSliding_4x4.INITIAL = InitialState                                          # Replace for a new puzzle
TileSliding_4x4.IsGoal = lambda s: s==(1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0)   # Immutable list
TileSliding_4x4.Actions = Actions
TileSliding_4x4.Result=Result
TileSliding_4x4.ActionCost = lambda s, a, sPrime: 1
print( TileSliding_4x4.IsGoal((1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,0)) )
print( Node(InitialState) )
print(1+TileSliding_4x4.ActionCost(1,2,3))

True
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0], <none>
2


Solution Function returns the solution.

In [248]:
"""
def Solution(node):
  if node.Parent==None:
    return []
  return Solution(node.Parent) + [node.Action]
"""

'\ndef Solution(node):\n  if node.Parent==None:\n    return []\n  return Solution(node.Parent) + [node.Action]\n'

In [249]:
def Solution(node):
    if node is None:
        return None      # or return [] and treat that as “no solution”
    if node.Parent is None:
        return []
    return Solution(node.Parent) + [node.Action]

# Create 15 3x3 Problems and 15 4x4 Problems

Generates 15 3x3 problems

In [250]:
setInitialStateSpaceSize(False)

problemList_3x3 = []
problemList_3x3_sol = []

for i in randomWalkSteps:
  print("Steps:", i)
  for j in range(3):
    state_3x3, solution_3x3 = RandomWalk(GoalState, i)
    problemList_3x3.append(state_3x3)
    problemList_3x3_sol.append(solution_3x3)
    PrintState(state_3x3)
    print (solution_3x3)
  print ()



Steps: 5
[2, 0, 3]
[1, 5, 6]
[4, 7, 8]
['l', 'l', 'u', 'u', 'r']
[4, 1, 2]
[0, 5, 3]
[7, 8, 6]
['u', 'u', 'l', 'l', 'd']
[1, 5, 2]
[4, 3, 0]
[7, 8, 6]
['u', 'u', 'l', 'd', 'r']

Steps: 10
[4, 1, 2]
[7, 6, 3]
[5, 8, 0]
['l', 'u', 'r', 'u', 'l', 'l', 'd', 'd', 'r', 'r']
[1, 2, 0]
[6, 8, 3]
[4, 7, 5]
['l', 'u', 'r', 'd', 'l', 'l', 'u', 'r', 'r', 'u']
[2, 3, 5]
[1, 0, 6]
[7, 4, 8]
['u', 'l', 'l', 'u', 'r', 'r', 'd', 'd', 'l', 'u']

Steps: 20
[0, 2, 3]
[8, 5, 4]
[7, 1, 6]
['l', 'l', 'u', 'u', 'r', 'd', 'd', 'l', 'u', 'r', 'd', 'r', 'u', 'l', 'd', 'l', 'u', 'r', 'u', 'l']
[5, 2, 3]
[4, 0, 6]
[7, 8, 1]
['l', 'l', 'u', 'u', 'r', 'd', 'l', 'u', 'r', 'r', 'd', 'l', 'l', 'd', 'r', 'r', 'u', 'u', 'l', 'd']
[8, 4, 2]
[7, 0, 5]
[6, 3, 1]
['u', 'l', 'd', 'r', 'u', 'u', 'l', 'l', 'd', 'r', 'u', 'l', 'd', 'd', 'r', 'u', 'r', 'd', 'l', 'u']

Steps: 40
[7, 4, 6]
[8, 0, 1]
[5, 3, 2]
['u', 'u', 'l', 'd', 'r', 'u', 'l', 'l', 'd', 'd', 'r', 'r', 'u', 'u', 'l', 'l', 'd', 'd', 'r', 'u', 'r', 'd', 'l', 'u', 'l'

Generates 15 4x4 problems

In [251]:
setInitialStateSpaceSize(True)

problemList_4x4 = []
problemList_4x4_sol = []

for i in randomWalkSteps:
  print("Steps:", i)
  for j in range(3):
    state_4x4, solution_4x4 = RandomWalk(GoalState, i)
    problemList_4x4.append(state_4x4)
    problemList_4x4_sol.append(solution_4x4)
    PrintState(state_4x4)
    print (solution_4x4)
  print ()

Steps: 5
[1, 2, 3, 4]
[5, 6, 7, 8]
[13, 9, 10, 12]
[0, 14, 11, 15]
['l', 'u', 'l', 'l', 'd']
[1, 2, 3, 4]
[5, 10, 6, 7]
[9, 0, 11, 8]
[13, 14, 15, 12]
['u', 'u', 'l', 'l', 'd']
[1, 0, 3, 4]
[5, 2, 6, 7]
[9, 10, 11, 8]
[13, 14, 15, 12]
['u', 'u', 'l', 'l', 'u']

Steps: 10
[1, 2, 3, 4]
[5, 6, 11, 7]
[13, 9, 0, 8]
[14, 15, 10, 12]
['u', 'u', 'l', 'd', 'l', 'l', 'd', 'r', 'r', 'u']
[1, 2, 3, 4]
[5, 11, 7, 8]
[9, 6, 0, 10]
[13, 14, 15, 12]
['u', 'u', 'l', 'd', 'l', 'u', 'r', 'r', 'd', 'l']
[0, 5, 3, 4]
[2, 1, 6, 8]
[9, 10, 7, 12]
[13, 14, 11, 15]
['l', 'u', 'u', 'l', 'u', 'l', 'd', 'r', 'u', 'l']

Steps: 20
[2, 6, 3, 4]
[1, 7, 8, 11]
[13, 5, 15, 12]
[9, 0, 10, 14]
['u', 'l', 'd', 'l', 'u', 'l', 'u', 'u', 'r', 'd', 'r', 'r', 'd', 'd', 'l', 'l', 'u', 'l', 'd', 'r']
[1, 3, 6, 4]
[5, 0, 11, 7]
[10, 2, 13, 12]
[9, 14, 8, 15]
['u', 'u', 'l', 'l', 'u', 'r', 'd', 'd', 'r', 'd', 'l', 'l', 'l', 'u', 'r', 'd', 'r', 'u', 'l', 'u']
[1, 2, 3, 4]
[5, 7, 8, 11]
[0, 6, 10, 14]
[13, 15, 9, 12]
['u', 'l', 'l'

Breadth First Search

# Puzzle Problems

## Breadth-First Search (Uninformed)

Breadth-First Search

15 3x3 Breadth-First Search

In [252]:
setInitialStateSpaceSize(False)

print("Breadth-First Search")

Solutions_BFS3x3 = []

print("")
for num, start_state in enumerate(problemList_3x3):
  TileSliding.INITIAL = start_state
  ret, cost = BreadthFirstSearch(TileSliding)
  if ret is None:
    sol = []
    print("Could not complete within the limit for expanded nodes.")
  else:
    sol = Solution(ret)
  Solutions_BFS3x3.append((start_state, sol, cost))



  print("Puzzle #", num + 1)
  print("Randomly Generated with", len(problemList_3x3_sol[num]), "steps.")
  print("Start State:")
  PrintState(start_state)
  print ("-----------------------")

  print("Solution:", sol)
  print("Length of Solution:", len(sol))
  print("Nodes Expanded: ", cost)
  print ("Random walk actions:", start_state)
  print ("-----------------------")
  print()

print("-------")
print (Solutions_BFS3x3)


Breadth-First Search

Puzzle # 1
Randomly Generated with 5 steps.
Start State:
[2, 0, 3]
[1, 5, 6]
[4, 7, 8]
-----------------------
Solution: ['l', 'd', 'd', 'r', 'r']
Length of Solution: 5
Nodes Expanded:  25
Random walk actions: [2, 0, 3, 1, 5, 6, 4, 7, 8]
-----------------------

Puzzle # 2
Randomly Generated with 5 steps.
Start State:
[4, 1, 2]
[0, 5, 3]
[7, 8, 6]
-----------------------
Solution: ['u', 'r', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  22
Random walk actions: [4, 1, 2, 0, 5, 3, 7, 8, 6]
-----------------------

Puzzle # 3
Randomly Generated with 5 steps.
Start State:
[1, 5, 2]
[4, 3, 0]
[7, 8, 6]
-----------------------
Solution: ['l', 'u', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  28
Random walk actions: [1, 5, 2, 4, 3, 0, 7, 8, 6]
-----------------------

Puzzle # 4
Randomly Generated with 10 steps.
Start State:
[4, 1, 2]
[7, 6, 3]
[5, 8, 0]
-----------------------
Solution: ['l', 'l', 'u', 'u', 'r', 'r', 'd', 'l', 'd', 'r']
Length of Soluti

Check BFS Solution code:

In [253]:
start, moves, cost = Solutions_BFS3x3[3]
if moves is None:
    print("No solution. Cannot ApplyMoves.")
else:
    end = ApplyMoves(moves, start.copy())
    PrintState(start)
    PrintState(end)

[4, 1, 2]
[7, 6, 3]
[5, 8, 0]
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]


15 4x4 Breadth-First Search problems

In [254]:
setInitialStateSpaceSize(True)

print("Breadth-First Search")

Solutions_BFS4x4 = []

print("")
for num, start_state in enumerate(problemList_4x4):
  TileSliding_4x4.INITIAL = start_state
  ret, cost = BreadthFirstSearch(TileSliding_4x4)
  if ret is None:
    sol = []
    print("Could not complete within the limit for expanded nodes.")
  else:
    sol = Solution(ret)
  Solutions_BFS4x4.append((start_state, sol, cost))



  print("Puzzle #", num + 1)
  print("Randomly Generated with", len(problemList_4x4_sol[num]), "steps.")
  print("Start State:")
  PrintState(start_state)
  print ("-----------------------")

  print("Solution:", sol)
  print("Length of Solution:", len(sol))
  print("Nodes Expanded: ", cost)
  print ("Random walk actions:", start_state)
  print ("-----------------------")
  print()

print("-------")
print (Solutions_BFS4x4)


Breadth-First Search

Puzzle # 1
Randomly Generated with 5 steps.
Start State:
[1, 2, 3, 4]
[5, 6, 7, 8]
[13, 9, 10, 12]
[0, 14, 11, 15]
-----------------------
Solution: ['u', 'r', 'r', 'd', 'r']
Length of Solution: 5
Nodes Expanded:  27
Random walk actions: [1, 2, 3, 4, 5, 6, 7, 8, 13, 9, 10, 12, 0, 14, 11, 15]
-----------------------

Puzzle # 2
Randomly Generated with 5 steps.
Start State:
[1, 2, 3, 4]
[5, 10, 6, 7]
[9, 0, 11, 8]
[13, 14, 15, 12]
-----------------------
Solution: ['u', 'r', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  47
Random walk actions: [1, 2, 3, 4, 5, 10, 6, 7, 9, 0, 11, 8, 13, 14, 15, 12]
-----------------------

Puzzle # 3
Randomly Generated with 5 steps.
Start State:
[1, 0, 3, 4]
[5, 2, 6, 7]
[9, 10, 11, 8]
[13, 14, 15, 12]
-----------------------
Solution: ['d', 'r', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  40
Random walk actions: [1, 0, 3, 4, 5, 2, 6, 7, 9, 10, 11, 8, 13, 14, 15, 12]
-----------------------

Puzzle # 4
Randomly Gene

In [270]:
start, moves, cost = Solutions_BFS4x4[3]
if moves is None:
    print("No solution. Cannot ApplyMoves.")
else:
    end = ApplyMoves(moves, start.copy())
    PrintState(start)
    PrintState(end)

[1, 2, 3, 4]
[5, 6, 11, 7]
[13, 9, 0, 8]
[14, 15, 10, 12]
[1, 2, 3, 4]
[5, 6, 7, 8]
[9, 10, 11, 12]
[13, 14, 15, 0]


Display Dataframe of 3x3 and 4x4 Breadth-First Search Problem Solutions

In [255]:
import pandas as pd

# Create a list to store the data for the DataFrame
data = []

# Process 3x3 solutions
for i, (start_state, solution, cost) in enumerate(Solutions_BFS3x3):
    data.append({
        'Puzzle #': i + 1,
        'Size': '3x3',
        'Start State': start_state,
        'Solution': solution,
        'Length of Solution': len(solution),
        'Nodes Expanded': cost
    })

# Process 4x4 solutions
for i, (start_state, solution, cost) in enumerate(Solutions_BFS4x4):
    data.append({
        'Puzzle #': i + 1,
        'Size': '4x4',
        'Start State': start_state,
        'Solution': solution,
        'Length of Solution': len(solution),
        'Nodes Expanded': cost
    })

# Create the DataFrame
df = pd.DataFrame(data)
df


,Puzzle #,Size,Start State,Solution,Length of Solution,Nodes Expanded
0,1,3x3,"[2, 0, 3, 1, 5, 6, 4, 7, 8]","[l, d, d, r, r]",5,25
1,2,3x3,"[4, 1, 2, 0, 5, 3, 7, 8, 6]","[u, r, r, d, d]",5,22
2,3,3x3,"[1, 5, 2, 4, 3, 0, 7, 8, 6]","[l, u, r, d, d]",5,28
3,4,3x3,"[4, 1, 2, 7, 6, 3, 5, 8, 0]","[l, l, u, u, r, r, d, l, d, r]",10,401
4,5,3x3,"[1, 2, 0, 6, 8, 3, 4, 7, 5]","[d, l, l, d, r, r, u, l, d, r]",10,344
5,6,3x3,"[2, 3, 5, 1, 0, 6, 7, 4, 8]","[d, r, u, u, l, l, d, r, r, d]",10,425
6,7,3x3,"[0, 2, 3, 8, 5, 4, 7, 1, 6]","[r, d, l, d, r, u, r, d, l, u, l, d, r, u, u, ...",20,34830
7,8,3x3,"[5, 2, 3, 4, 0, 6, 7, 8, 1]","[u, l, d, r, u, r, d, d, l, u, r, u, l, d, l, ...",20,33421
8,9,3x3,"[8, 4, 2, 7, 0, 5, 6, 3, 1]","[d, r, u, l, d, l, u, u, r, d, l, u, r, r, d, ...",20,38958
9,10,3x3,"[7, 4, 6, 8, 0, 1, 5, 3, 2]","[d, l, u, u, r, d, r, u, l, d, d, r, u, l, d, ...",22,70227


## AStar using Manhattan Distance (Informed)

A* using Manhattan Distance combines the path cost with the estimated cost to the goal.
f(n) = g(n) + h(n)

Essentially Astar takes an uninformed algorithm and incorporates a heuristic by summing it with the path cost.

f(n) becomes the priority function.

The heuristic need to be scaled properly so that it underestimates the true cost. h(n) is smaller than h*(n). So,
h* is admissible if it doesn't overestimate the goal

In [256]:
setInitialStateSpaceSize(False)
AStarFb_MD_3x3 = lambda n: n.PathCost + ManhattanDistance(n.State, GoalState)

15 3x3 A* Manhattan

In [257]:
setInitialStateSpaceSize(False)

print("A* using Manhattan Distance")

Solutions_MD3x3 = []

print("")
for num, start_state in enumerate(problemList_3x3):
  TileSliding.INITIAL = start_state
  ret, cost = BestFirstSearch(TileSliding, AStarFb_MD_3x3)
  if ret is None:
    sol = []
    print("Could not complete within the limit for expanded nodes.")
  else:
    sol = Solution(ret)
  Solutions_MD3x3.append((start_state, sol, cost))




  print("Puzzle #", num + 1)
  print("Randomly Generated with", len(problemList_3x3_sol[num]), "steps.")
  print("Start State:")
  PrintState(start_state)
  print ("-----------------------")

  print("Solution:", sol)
  print("Length of Solution:", len(sol))
  print("Nodes Expanded: ", cost)
  print ("Random walk actions:", start_state)
  print ("-----------------------")
  print ()

print("-------")
print (Solutions_MD3x3)


A* using Manhattan Distance

Puzzle # 1
Randomly Generated with 5 steps.
Start State:
[2, 0, 3]
[1, 5, 6]
[4, 7, 8]
-----------------------
Solution: ['l', 'd', 'd', 'r', 'r']
Length of Solution: 5
Nodes Expanded:  5
Random walk actions: [2, 0, 3, 1, 5, 6, 4, 7, 8]
-----------------------

Puzzle # 2
Randomly Generated with 5 steps.
Start State:
[4, 1, 2]
[0, 5, 3]
[7, 8, 6]
-----------------------
Solution: ['u', 'r', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  5
Random walk actions: [4, 1, 2, 0, 5, 3, 7, 8, 6]
-----------------------

Puzzle # 3
Randomly Generated with 5 steps.
Start State:
[1, 5, 2]
[4, 3, 0]
[7, 8, 6]
-----------------------
Solution: ['l', 'u', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  6
Random walk actions: [1, 5, 2, 4, 3, 0, 7, 8, 6]
-----------------------

Puzzle # 4
Randomly Generated with 10 steps.
Start State:
[4, 1, 2]
[7, 6, 3]
[5, 8, 0]
-----------------------
Solution: ['l', 'l', 'u', 'u', 'r', 'r', 'd', 'l', 'd', 'r']
Length of So

In [ ]:
# Test Solution with ApplyModes

start, moves, cost = Solutions_MD3x3[3]
if moves is None:
    print("No solution. Cannot ApplyMoves.")
else:
    end = ApplyMoves(moves, start.copy())
    PrintState(start)
    PrintState(end)

[4, 1, 2]
[7, 6, 3]
[5, 8, 0]
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]


15 4x4 A* Manhattan Distance problems

In [258]:
setInitialStateSpaceSize(True)
AStarFb_MD_4x4 = lambda n: n.PathCost + ManhattanDistance(n.State, GoalState)

In [259]:
setInitialStateSpaceSize(True)

print("A* with Manhattan Distance Heuristic")
print ("-----------------------")

Solutions_MD4x4 = []

print("")
for num, start_state in enumerate(problemList_4x4):
  TileSliding_4x4.INITIAL = start_state
  ret, cost = BestFirstSearch(TileSliding_4x4, AStarFb_MD_4x4)
  if ret is None:
    sol = []
    print("Could not complete within the limit for expanded nodes.")
  else:
    sol = Solution(ret)
  Solutions_MD4x4.append((start_state, sol, cost))



  print("Puzzle #", num + 1)
  print("Randomly Generated with", len(problemList_4x4_sol[num]), "steps.")
  print("Start State:")
  PrintState(start_state)
  print ("-----------------------")

  print("Solution:", sol)
  print("Length of Solution:", len(sol))
  print("Nodes Expanded: ", cost)
  print ("Random walk actions:", start_state)
  print ("-----------------------")

print("-------")
print (Solutions_MD4x4)


A* with Manhattan Distance Heuristic
-----------------------

Puzzle # 1
Randomly Generated with 5 steps.
Start State:
[1, 2, 3, 4]
[5, 6, 7, 8]
[13, 9, 10, 12]
[0, 14, 11, 15]
-----------------------
Solution: ['u', 'r', 'r', 'd', 'r']
Length of Solution: 5
Nodes Expanded:  5
Random walk actions: [1, 2, 3, 4, 5, 6, 7, 8, 13, 9, 10, 12, 0, 14, 11, 15]
-----------------------
Puzzle # 2
Randomly Generated with 5 steps.
Start State:
[1, 2, 3, 4]
[5, 10, 6, 7]
[9, 0, 11, 8]
[13, 14, 15, 12]
-----------------------
Solution: ['u', 'r', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  5
Random walk actions: [1, 2, 3, 4, 5, 10, 6, 7, 9, 0, 11, 8, 13, 14, 15, 12]
-----------------------
Puzzle # 3
Randomly Generated with 5 steps.
Start State:
[1, 0, 3, 4]
[5, 2, 6, 7]
[9, 10, 11, 8]
[13, 14, 15, 12]
-----------------------
Solution: ['d', 'r', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  5
Random walk actions: [1, 0, 3, 4, 5, 2, 6, 7, 9, 10, 11, 8, 13, 14, 15, 12]
--------------

In [ ]:
# Test Solution with ApplyModes

start, moves, cost = Solutions_MD4x4[3]
if moves is None:
    print("No solution. Cannot ApplyMoves.")
else:
    end = ApplyMoves(moves, start.copy())
    PrintState(start)
    PrintState(end)

[4, 1, 2]
[7, 6, 3]
[5, 8, 0]
[1, 2, 3]
[4, 5, 6]
[7, 8, 0]


In [260]:
# Create a list to store the data for the DataFrame
data_manhattan = []

# Process 3x3 solutions
for i, (start_state, solution, cost) in enumerate(Solutions_MD3x3):
    data_manhattan.append({
        'Puzzle #': i + 1,
        'Size': '3x3',
        'Start State': start_state,
        'Solution': solution,
        'Length of Solution': len(solution),
        'Nodes Expanded': cost
    })

# Process 4x4 solutions
for i, (start_state, solution, cost) in enumerate(Solutions_MD4x4):
    data_manhattan.append({
        'Puzzle #': i + 1,
        'Size': '4x4',
        'Start State': start_state,
        'Solution': solution,
        'Length of Solution': len(solution),
        'Nodes Expanded': cost
    })

# Create the DataFrame
df_manhattan = pd.DataFrame(data_manhattan)

print("A* using Manhattan Distance Heuristic for 3x3 and 4x4 Puzzles:")
df_manhattan


A* using Manhattan Distance Heuristic for 3x3 and 4x4 Puzzles:


,Puzzle #,Size,Start State,Solution,Length of Solution,Nodes Expanded
0,1,3x3,"[2, 0, 3, 1, 5, 6, 4, 7, 8]","[l, d, d, r, r]",5,5
1,2,3x3,"[4, 1, 2, 0, 5, 3, 7, 8, 6]","[u, r, r, d, d]",5,5
2,3,3x3,"[1, 5, 2, 4, 3, 0, 7, 8, 6]","[l, u, r, d, d]",5,6
3,4,3x3,"[4, 1, 2, 7, 6, 3, 5, 8, 0]","[l, l, u, u, r, r, d, l, d, r]",10,12
4,5,3x3,"[1, 2, 0, 6, 8, 3, 4, 7, 5]","[d, l, l, d, r, r, u, l, d, r]",10,17
5,6,3x3,"[2, 3, 5, 1, 0, 6, 7, 4, 8]","[d, r, u, u, l, l, d, r, r, d]",10,21
6,7,3x3,"[0, 2, 3, 8, 5, 4, 7, 1, 6]","[r, d, l, d, r, u, r, d, l, u, l, d, r, u, u, ...",20,636
7,8,3x3,"[5, 2, 3, 4, 0, 6, 7, 8, 1]","[l, u, r, d, l, d, r, r, u, l, d, l, u, r, u, ...",20,1031
8,9,3x3,"[8, 4, 2, 7, 0, 5, 6, 3, 1]","[d, r, u, l, d, l, u, u, r, d, l, u, r, r, d, ...",20,182
9,10,3x3,"[7, 4, 6, 8, 0, 1, 5, 3, 2]","[l, u, r, r, d, d, l, u, u, r, d, d, l, l, u, ...",22,857


## AStar using OutOfPlace (Informed)

In [261]:
setInitialStateSpaceSize(False)
AStarFb_OOP_3x3 = lambda n: n.PathCost + OutOfPlace(n.State, GoalState)

15 3x3 A* Using Out Of Place

In [262]:
setInitialStateSpaceSize(False)

print("A* using Out of Place")

Solutions_OOP3x3 = []

print("")
for num, start_state in enumerate(problemList_3x3):
  TileSliding.INITIAL = start_state
  ret, cost = BestFirstSearch(TileSliding, AStarFb_OOP_3x3)
  if ret is None:
    sol = []
    print("Could not complete within the limit for expanded nodes.")
  else:
    sol = Solution(ret)
  Solutions_OOP3x3.append((start_state, sol, cost))



  print("Puzzle #", num + 1)
  print("Randomly Generated with", len(problemList_3x3_sol[num]), "steps.")
  print("Start State:")
  PrintState(start_state)
  print ("-----------------------")

  print("Solution:", sol)
  print("Length of Solution:", len(sol))
  print("Nodes Expanded: ", cost)
  print ("Random walk actions:", start_state)
  print ("-----------------------")
  print ()

print("-------")
print (Solutions_OOP3x3)


A* using Out of Place

Puzzle # 1
Randomly Generated with 5 steps.
Start State:
[2, 0, 3]
[1, 5, 6]
[4, 7, 8]
-----------------------
Solution: ['l', 'd', 'd', 'r', 'r']
Length of Solution: 5
Nodes Expanded:  5
Random walk actions: [2, 0, 3, 1, 5, 6, 4, 7, 8]
-----------------------

Puzzle # 2
Randomly Generated with 5 steps.
Start State:
[4, 1, 2]
[0, 5, 3]
[7, 8, 6]
-----------------------
Solution: ['u', 'r', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  5
Random walk actions: [4, 1, 2, 0, 5, 3, 7, 8, 6]
-----------------------

Puzzle # 3
Randomly Generated with 5 steps.
Start State:
[1, 5, 2]
[4, 3, 0]
[7, 8, 6]
-----------------------
Solution: ['l', 'u', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  7
Random walk actions: [1, 5, 2, 4, 3, 0, 7, 8, 6]
-----------------------

Puzzle # 4
Randomly Generated with 10 steps.
Start State:
[4, 1, 2]
[7, 6, 3]
[5, 8, 0]
-----------------------
Solution: ['l', 'l', 'u', 'u', 'r', 'r', 'd', 'l', 'd', 'r']
Length of Solution

In [3]:
# Test Solution with ApplyModes

start, moves, cost = Solutions_OOP3x3[3]
if moves is None:
    print("No solution. Cannot ApplyMoves.")
else:
    end = ApplyMoves(moves, start.copy())
    PrintState(start)
    PrintState(end)


'\nstart, moves, cost = Solutions_OOP3x3[3]\nif moves is None:\n    print("No solution. Cannot ApplyMoves.")\nelse:\n    end = ApplyMoves(moves, start.copy())\n    PrintState(start)\n    PrintState(end)\n    '

15 4x4 A* using Out Of Place

In [263]:
setInitialStateSpaceSize(True)
AStarFb_OOP_4x4 = lambda n: n.PathCost + OutOfPlace(n.State, GoalState)

In [264]:
setInitialStateSpaceSize(True)

print("A* Out of Place")

Solutions_OOP4x4 = []

print("")
for num, start_state in enumerate(problemList_4x4):
  TileSliding_4x4.INITIAL = start_state
  ret, cost = BestFirstSearch(TileSliding_4x4, AStarFb_OOP_4x4)
  if ret is None:
    sol = []
    print("Could not complete within the limit for expanded nodes.")
  else:
    sol = Solution(ret)
  Solutions_OOP4x4.append((start_state, sol, cost))



  print("Puzzle #", num + 1)
  print("Randomly Generated with", len(problemList_4x4_sol[num]), "steps.")
  print("Start State:")
  PrintState(start_state)
  print ("-----------------------")

  print("Solution:", sol)
  print("Length of Solution:", len(sol))
  print("Nodes Expanded: ", cost)
  print ("Random walk actions:", start_state)
  print ("-----------------------")

print("-------")
print (Solutions_OOP4x4)


A* Out of Place

Puzzle # 1
Randomly Generated with 5 steps.
Start State:
[1, 2, 3, 4]
[5, 6, 7, 8]
[13, 9, 10, 12]
[0, 14, 11, 15]
-----------------------
Solution: ['u', 'r', 'r', 'd', 'r']
Length of Solution: 5
Nodes Expanded:  5
Random walk actions: [1, 2, 3, 4, 5, 6, 7, 8, 13, 9, 10, 12, 0, 14, 11, 15]
-----------------------
Puzzle # 2
Randomly Generated with 5 steps.
Start State:
[1, 2, 3, 4]
[5, 10, 6, 7]
[9, 0, 11, 8]
[13, 14, 15, 12]
-----------------------
Solution: ['u', 'r', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  5
Random walk actions: [1, 2, 3, 4, 5, 10, 6, 7, 9, 0, 11, 8, 13, 14, 15, 12]
-----------------------
Puzzle # 3
Randomly Generated with 5 steps.
Start State:
[1, 0, 3, 4]
[5, 2, 6, 7]
[9, 10, 11, 8]
[13, 14, 15, 12]
-----------------------
Solution: ['d', 'r', 'r', 'd', 'd']
Length of Solution: 5
Nodes Expanded:  5
Random walk actions: [1, 0, 3, 4, 5, 2, 6, 7, 9, 10, 11, 8, 13, 14, 15, 12]
-----------------------
Puzzle # 4
Randomly Generated with 

Display AStar using Out of Place dataframe:

In [265]:
import pandas as pd

# Assuming Solutions_OOP3x3 and Solutions_OOP4x4 are defined as in the original code

# Create a list to store the data for the DataFrame
data_oop = []

# Process 3x3 solutions
for i, (start_state, solution, cost) in enumerate(Solutions_OOP3x3):
    data_oop.append({
        'Puzzle #': i + 1,
        'Size': '3x3',
        'Start State': start_state,
        'Solution': solution,
        'Length of Solution': len(solution),
        'Nodes Expanded': cost
    })

# Process 4x4 solutions
for i, (start_state, solution, cost) in enumerate(Solutions_OOP4x4):
    data_oop.append({
        'Puzzle #': i + 1,
        'Size': '4x4',
        'Start State': start_state,
        'Solution': solution,
        'Length of Solution': len(solution),
        'Nodes Expanded': cost
    })

# Create the DataFrame
df_oop = pd.DataFrame(data_oop)
df_oop


,Puzzle #,Size,Start State,Solution,Length of Solution,Nodes Expanded
0,1,3x3,"[2, 0, 3, 1, 5, 6, 4, 7, 8]","[l, d, d, r, r]",5,5
1,2,3x3,"[4, 1, 2, 0, 5, 3, 7, 8, 6]","[u, r, r, d, d]",5,5
2,3,3x3,"[1, 5, 2, 4, 3, 0, 7, 8, 6]","[l, u, r, d, d]",5,7
3,4,3x3,"[4, 1, 2, 7, 6, 3, 5, 8, 0]","[l, l, u, u, r, r, d, l, d, r]",10,25
4,5,3x3,"[1, 2, 0, 6, 8, 3, 4, 7, 5]","[d, l, l, d, r, r, u, l, d, r]",10,38
5,6,3x3,"[2, 3, 5, 1, 0, 6, 7, 4, 8]","[d, r, u, u, l, l, d, r, r, d]",10,46
6,7,3x3,"[0, 2, 3, 8, 5, 4, 7, 1, 6]","[r, d, l, d, r, u, r, d, l, u, l, d, r, u, u, ...",20,3533
7,8,3x3,"[5, 2, 3, 4, 0, 6, 7, 8, 1]","[u, r, d, d, l, l, u, r, r, u, l, l, d, r, u, ...",20,4541
8,9,3x3,"[8, 4, 2, 7, 0, 5, 6, 3, 1]","[d, r, u, l, d, l, u, r, u, l, d, r, u, r, d, ...",20,3630
9,10,3x3,"[7, 4, 6, 8, 0, 1, 5, 3, 2]","[d, l, u, u, r, d, r, u, l, d, d, r, u, l, d, ...",22,9426


In [266]:
# Test Solution by applying the action sequence found by the A* Out of Place search to the initial state
start, moves, cost = Solutions_OOP4x4[11]
if moves is None:
    print("No solution. Cannot ApplyMoves.")
else:
    end = ApplyMoves(moves, start.copy())
    PrintState(start)
    PrintState(end)

[5, 1, 0, 2]
[13, 3, 8, 4]
[10, 11, 6, 12]
[9, 14, 15, 7]
[5, 1, 0, 2]
[13, 3, 8, 4]
[10, 11, 6, 12]
[9, 14, 15, 7]


## Display all Dataframes

The breadth remains the same between the two domains of 3x3 and 4x4. However the depth increases significantly in the jump to a higher dimension.

In [267]:
print("Breadth-First Search")
df
df

,Puzzle #,Size,Start State,Solution,Length of Solution,Nodes Expanded
0,1,3x3,"[2, 0, 3, 1, 5, 6, 4, 7, 8]","[l, d, d, r, r]",5,25
1,2,3x3,"[4, 1, 2, 0, 5, 3, 7, 8, 6]","[u, r, r, d, d]",5,22
2,3,3x3,"[1, 5, 2, 4, 3, 0, 7, 8, 6]","[l, u, r, d, d]",5,28
3,4,3x3,"[4, 1, 2, 7, 6, 3, 5, 8, 0]","[l, l, u, u, r, r, d, l, d, r]",10,401
4,5,3x3,"[1, 2, 0, 6, 8, 3, 4, 7, 5]","[d, l, l, d, r, r, u, l, d, r]",10,344
5,6,3x3,"[2, 3, 5, 1, 0, 6, 7, 4, 8]","[d, r, u, u, l, l, d, r, r, d]",10,425
6,7,3x3,"[0, 2, 3, 8, 5, 4, 7, 1, 6]","[r, d, l, d, r, u, r, d, l, u, l, d, r, u, u, ...",20,34830
7,8,3x3,"[5, 2, 3, 4, 0, 6, 7, 8, 1]","[u, l, d, r, u, r, d, d, l, u, r, u, l, d, l, ...",20,33421
8,9,3x3,"[8, 4, 2, 7, 0, 5, 6, 3, 1]","[d, r, u, l, d, l, u, u, r, d, l, u, r, r, d, ...",20,38958
9,10,3x3,"[7, 4, 6, 8, 0, 1, 5, 3, 2]","[d, l, u, u, r, d, r, u, l, d, d, r, u, l, d, ...",22,70227


In [268]:
print("A* using Manhattan Distance")
df_manhattan

,Puzzle #,Size,Start State,Solution,Length of Solution,Nodes Expanded
0,1,3x3,"[2, 0, 3, 1, 5, 6, 4, 7, 8]","[l, d, d, r, r]",5,5
1,2,3x3,"[4, 1, 2, 0, 5, 3, 7, 8, 6]","[u, r, r, d, d]",5,5
2,3,3x3,"[1, 5, 2, 4, 3, 0, 7, 8, 6]","[l, u, r, d, d]",5,6
3,4,3x3,"[4, 1, 2, 7, 6, 3, 5, 8, 0]","[l, l, u, u, r, r, d, l, d, r]",10,12
4,5,3x3,"[1, 2, 0, 6, 8, 3, 4, 7, 5]","[d, l, l, d, r, r, u, l, d, r]",10,17
5,6,3x3,"[2, 3, 5, 1, 0, 6, 7, 4, 8]","[d, r, u, u, l, l, d, r, r, d]",10,21
6,7,3x3,"[0, 2, 3, 8, 5, 4, 7, 1, 6]","[r, d, l, d, r, u, r, d, l, u, l, d, r, u, u, ...",20,636
7,8,3x3,"[5, 2, 3, 4, 0, 6, 7, 8, 1]","[l, u, r, d, l, d, r, r, u, l, d, l, u, r, u, ...",20,1031
8,9,3x3,"[8, 4, 2, 7, 0, 5, 6, 3, 1]","[d, r, u, l, d, l, u, u, r, d, l, u, r, r, d, ...",20,182
9,10,3x3,"[7, 4, 6, 8, 0, 1, 5, 3, 2]","[l, u, r, r, d, d, l, u, u, r, d, d, l, l, u, ...",22,857


In [269]:
print ("A* using Out of Place")
df_oop

,Puzzle #,Size,Start State,Solution,Length of Solution,Nodes Expanded
0,1,3x3,"[2, 0, 3, 1, 5, 6, 4, 7, 8]","[l, d, d, r, r]",5,5
1,2,3x3,"[4, 1, 2, 0, 5, 3, 7, 8, 6]","[u, r, r, d, d]",5,5
2,3,3x3,"[1, 5, 2, 4, 3, 0, 7, 8, 6]","[l, u, r, d, d]",5,7
3,4,3x3,"[4, 1, 2, 7, 6, 3, 5, 8, 0]","[l, l, u, u, r, r, d, l, d, r]",10,25
4,5,3x3,"[1, 2, 0, 6, 8, 3, 4, 7, 5]","[d, l, l, d, r, r, u, l, d, r]",10,38
5,6,3x3,"[2, 3, 5, 1, 0, 6, 7, 4, 8]","[d, r, u, u, l, l, d, r, r, d]",10,46
6,7,3x3,"[0, 2, 3, 8, 5, 4, 7, 1, 6]","[r, d, l, d, r, u, r, d, l, u, l, d, r, u, u, ...",20,3533
7,8,3x3,"[5, 2, 3, 4, 0, 6, 7, 8, 1]","[u, r, d, d, l, l, u, r, r, u, l, l, d, r, u, ...",20,4541
8,9,3x3,"[8, 4, 2, 7, 0, 5, 6, 3, 1]","[d, r, u, l, d, l, u, r, u, l, d, r, u, r, d, ...",20,3630
9,10,3x3,"[7, 4, 6, 8, 0, 1, 5, 3, 2]","[d, l, u, u, r, d, r, u, l, d, d, r, u, l, d, ...",22,9426


# Analysis and Conclusion

This program extended the number of states in the 3x3 sliding puzzle to a 4x4 sliding puzzle. The added depth and complexity in combination with a larger step amount resulted in ballooning time and space complexity with the more computationally intensive search methods. This program tested 15 problems on the 3x3 sliding puzzle and 15 problems 4x4 on the sliding puzzle. After every 3 problems, the amount of steps doubled. All of the 3x3 problems were able to be solved with relative ease by all the algorithms. However, the search algorithms struggled much more with the 4x4 problems and some could not reach the goal state within the node expansion cutoff of 500,000. One uninformed search method and two informed methods were tested in this program: Breadth First Search, A* using Manhattan Distance, and A* using Out of Place.      

Comparing the results between these search algorithms, Breadth First Search (BFS) performs the worst from a performance perspective. As the steps performed by random walk increased, the amount of computation increased. BFS struggles at greater depths because its time and memory complexity is O(b^d). Where b is the breadth and d is the depth of the solution. Because BFS expands all nodes in FIFO order, it must expand an exponentially growing number of nodes before going into greater depths. So BFS struggles with more complex problems like the 4x4 variation of the sliding puzzle that require more involved exploration before reaching the goal state. This is compounded by the fact that it's a uninformed algorithm where it lacks any supporting, contextual information like heuristics.

In this program, BFS is shown to be a wildly impractical computational search algorithm with problems generated with 20 steps or more on the 4x4 sliding puzzle. The branching factor remains relatively constant, so the escalation in complexity is driven by increasing the depth of the solution. The branches of the search tree stay between 2 and 4. And since BFS exhausts every node before diving deeper into a tree, this search algorithm attempts to find a solution at a shallow depth level.  Even so, BFS's focus on finding the shortest route through trial and error causes it to expand more nodes whilst in order to reach the goal state.

A* using Manhattan Distance performed the best out of the three. The inclusion of Manhattan Distance heuristic function improved the efficiency of the base priority queue algorithm: Best First Search. Heuristic functions give you a general measure of the distance from the current state to the goal state. The manhattan distance is a general estimate used to determine the number of actions needed to get to the goal state. In this code, a single tile is considered at a time.    

A* using Out of Place performed somewhere in the middle between the other two. This search is better than Best First Search because it incorporates the heuristic for the number of out of place tiles. However, Out of Place is an inferior, less accurate measure of the distance to the ultimate goal than Manhattan Distance. Out of place only calculates the amount of misplaced tiles.  

All of the 3x3 sliding tile puzzles were able to be solved by all three search algorithms. As a baseline, these results for the 3x3 problems show how each algorithm performed with a simpler domain. The solution given by each search implementation varied significantly when he steps of the Random Walk increased. While all saw similar increases in the length of the solution, the search methods all differed in the amount that the nodes expanded: A* with Manhattan Distance saw far outperformed both models with problem #14 having the highest nodes extended requiring only 3952 nodes extended. A* using Out of Place was worse overall with some standout problems like #14 with 37971 nodes extended. Breadth-First Search performed even worse with problem #14 reaching 145810.

Unlike the 3x3 sliding tile puzzles however, all of the search algorithms encountered trouble a few of the 4x4 sliding tile problems. A* using Manhattan Distance managed to reach the goal state in 14 out of 15 problems within 500,000 node expansions. While the A* using Out of Place solved 10 out 15 problems. And the Breadth First Search could only solve a measly 6 out of 15 problems. Both the A* using Out of Place and Breadth First Search node extentions experienced a sudden increase in nodes extended which rose well above the 500,000 threshold. Although the jump appeared much earlier in the Breadth First Search algorithm. The exponential increase in BFS is especially surprising since the nodes extended in the problems that were able to be solved were still relatively low with problem #5 only reaching 2882 nodes extended.

In both the 3x3 and 4x4 sliding tile problems, A* using Manhattan Distance search function performed significantly better than either of the other two search algorithms. No matter the complexity of the problem, whether the problem was shallow or deep, A* using Manhattan Distance is the most efficient of the three search algorithms no matter the scale of the problem or domain.      

In the general domain of Artificial Intelligence search algorithms have fallen out of favor. However, even with the renewed focus on data-based algorithms, this program shows that search tools can remain relevant. Search algorithms can still benefit AI in situations where data is invalid, incompatible, or just doesn't exist like blindspots in data, optimization, or exploration of uncharted domains. This program examined how search can find the solution to a tile sliding puzzle either through brute force with the uninformed search method or with some understanding of the states through the informed search methods.

Beyond games and puzzles, Search is applicable to an even wider variety of domains. For example, A* with the Manhattan Distance heurisic function can easily be applied to navigation to find the quickest way from one location to another. This program has also shown the pitfalls of search algorithms in that they aren't always that efficient. This is especially true when it comes to more complex domains like the jump from 3x3 to 4x4 tiles. However, despite the shortcomings of uninformed search algorithms like Breadth First Search or Depth First Search with complex problems, informed algorithms like A* with Manhattan Distance hinted at the possibilities of incorporating search tools into Artificial Intelligence when data is not available.  